# Exercise 1: Unicode investigation

*Scripts and Encoding · version 0.1.0*

> Run `"İstanbul".lower()`, `"ISPARTA".lower()` and `turkish_lower_nfc("I\u0307STANBUL")` from the Unicode section. Print `repr`, `len` and the code points of each result. Explain why the standard `str.lower()` is incorrect for Turkish, and why the helper normalises to NFC first.

**Setup.** Standard library only.

In [ ]:
import unicodedata

def turkish_lower_nfc(text: str) -> str:
    """Turkish/Azerbaijani lowercasing for ordinary text; returns an NFC view (from the chapter)."""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(str.maketrans({"İ": "i", "I": "ı"}))
    return unicodedata.normalize("NFC", text.lower())

def show(label, s):
    points = " ".join(f"U+{ord(c):04X}" for c in s)
    print(f"{label:<34} {s!r:<14} len={len(s):<3} {points}")

show('"İstanbul".lower()', "İstanbul".lower())
show('"ISPARTA".lower()', "ISPARTA".lower())
show('turkish_lower_nfc("I\\u0307STANBUL")', turkish_lower_nfc("I\u0307STANBUL"))
show('turkish_lower_nfc("İstanbul")', turkish_lower_nfc("İstanbul"))
show('turkish_lower_nfc("ISPARTA")', turkish_lower_nfc("ISPARTA"))
print()
for c in "\u0130I\u0307iı":
    print(f"U+{ord(c):04X}  {unicodedata.name(c)}")

## Worked solution

| Call | Result | `len` | Code points |
|---|---|---|---|
| `"İstanbul".lower()` | `'i̇stanbul'` | 9 | `i` U+0069 + U+0307 COMBINING DOT ABOVE + `stanbul` |
| `"ISPARTA".lower()` | `'isparta'` | 7 | dotted `i` U+0069 at the start |
| `turkish_lower_nfc("I\u0307STANBUL")` | `'istanbul'` | 8 | plain `i` U+0069 |

**Why `str.lower()` is wrong for Turkish.** Python applies the language-independent Unicode case mappings. Under them, capital dotless *I* lowercases to dotted *i* (Turkish needs *ı*), and capital *İ* (U+0130) lowercases to *i* plus a combining dot above, so the string even grows by one code point. Both results look plausible on screen but no longer match the Turkish word forms in a vocabulary.

**Why NFC first.** *İ* can also be written as *I* followed by U+0307. The helper's mapping `{"İ": "i", "I": "ı"}` would turn that decomposed *I* into dotless *ı* and leave a stray combining dot. NFC first composes *I* + U+0307 into the single code point U+0130, which the mapping then handles correctly; NFC at the end recomposes anything that `lower()` produced.